# Metric Definition

This notebook defines the operational metrics used in the project.

It loads the canonical scheduler dataset and the aggregated GPU telemetry dataset produced during the EDA stage, applies the required population filtering and preprocessing, and computes the selected metrics:

- North Star: weekly P95 turnaround time
- Guardrail 1: weekly failure / non-completion rate
- Guardrail 2: weekly median GPU utilization

The outputs of this notebook are saved as job-level and weekly metric tables, which will be used in the metric validation stage.

## Business Context

This project analyzes operational data from a High-Performance Computing (HPC) cluster. The system is managed by a job scheduler (Slurm) and serves multiple users submitting computational workloads. Some jobs use CPUs only, while others use GPUs for accelerated workloads such as AI/ML training.

In this context, the compute platform behaves like an internal “infrastructure-as-a-service” product. Its goals are:

- Deliver compute jobs reliably

- Minimize user waiting time

- Use expensive GPU resources efficiently

- Maintain system stability under variable demand

Users submit jobs. The scheduler queues and allocates resources. GPU telemetry records actual hardware utilization and energy usage during execution.

The central operational question becomes:

**How do we measure platform performance in a way that is stable, reliable, unbiased, and resistant to manipulation?**

This is where metric validity becomes critical.

### Why Metric Validity Matters Here

Operational metrics in infrastructure systems are often:

- Heavy-tailed

- Highly segmented (by job size, partition, workload type)

- Subject to drift over time

- Sensitive to rare congestion events

- Influenced by missing data

- Choosing the wrong metric can lead to:

- False conclusions

- Misleading improvements

- Gaming behavior

- Masked inefficiencies

Therefore, this project selects one North Star metric and two Guardrail metrics, then systematically evaluates their validity across:

- **Missingness, Segmentation bias, Drift, Stability, Reliability, Sensitivity, CUPED variance reduction**

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from pathlib import Path

## Read dataset & preprocessing steps

The following preprocessing steps were deducted based on the information obtained from previous EDA step.

In [2]:

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)

# --- Read datasets ---
sched_canon = pd.read_parquet(INTERIM_DIR / "scheduler_canonical.parquet")
dcgm_job = pd.read_parquet(INTERIM_DIR / "dcgm_job_level.parquet")

print("sched_canon:", sched_canon.shape)
print("dcgm_job:", dcgm_job.shape)

display(sched_canon.head())
display(dcgm_job.head())

sched_canon: (287069, 31)
dcgm_job: (74849, 5)


,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_req,gres_alloc,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type
0,61134,41161693674,4595979483,41415979807,51671871839,1,['r7269000-n717709'],1,0,0,gpu:1,gpu:1,NaN,0,0,xeon-g6,8,51200,gpu,10362,3,240,13633668,13633668,13633668,13635407,-1,0,"1=1,2=51200,4=1,5=1,1001=1","1=1,2=51200,4=1,5=1,1001=1",OTHER
1,246873,41161693674,4595979483,41415979807,51671871839,1,['r8532701-n772143'],1,256,256,gpu:1,gpu:1,NaN,0,0,xeon-g6,4,10240,gpu,10676,5,240,7825336,7825348,7825351,7825351,-1,0,"1=1,2=10240,4=1,5=1,1001=1","1=1,2=10240,4=1,5=1,1001=1",OTHER
2,325713,41161693674,4595979483,21074900298,51671871839,1,['r9659953-n750018'],2,0,0,gpu:volta:2,gpu:2,NaN,0,0,xeon-g6,2,9223372036854785408,gaia,10317,3,4294967295,13416154,13416154,13416154,13416181,-1,0,"1=2,2=19200,4=1,5=2,1002=2","1=2,2=19200,4=1,5=2,1002=2",OTHER
3,1055011,41161693674,4595979483,66088413977,51671871839,1,['r368967-n172107'],1,0,0,gpu:volta:1,gpu:1,NaN,0,0,xeon-g6,4,9223372036854785408,gaia,10213,3,5,24431626,24431626,24431627,24431627,-1,0,"1=1,2=9600,4=1,5=1,1002=1","1=1,2=9600,4=1,5=1,1002=1",OTHER
4,1218085,41161693674,4595979483,41415979807,41415979807,0,[],1,0,0,NaN,NaN,NaN,0,0,xeon-g6,0,10240,gpu,10690,4,240,7825124,-1,-1,7825250,-1,0,NaN,"1=1,2=10240,4=1,5=1,1001=1",OTHER


,id_job,dcgm_rows,energy_j,gpu_exec_s,sm_util_mean
0,1055011,1,0.0,0.55,0.0
1,1363534,1,781482.0,12904.10,96.0
2,1808972,1,73030.0,994.06,27.0
3,2688926,2,2551.0,99.52,0.0
4,3084235,1,130876.0,126303.00,1.0


### 1.Inspect columns and dtypes

In [3]:
print("sched_canon columns:")
print(sched_canon.columns.tolist())

print("\ndcgm_job columns:")
print(dcgm_job.columns.tolist())

print("\nScheduler dtypes:")
print(sched_canon.dtypes)

print("\nDCGM dtypes:")
print(dcgm_job.dtypes)

sched_canon columns:
['id_job', 'id_array_job', 'id_array_task', 'id_user', 'kill_requid', 'nodes_alloc', 'nodelist', 'cpus_req', 'derived_ec', 'exit_code', 'gres_req', 'gres_alloc', 'gres_used', 'array_max_tasks', 'array_task_pending', 'constraints', 'flags', 'mem_req', 'partition', 'priority', 'state', 'timelimit', 'time_submit', 'time_eligible', 'time_start', 'time_end', 'time_suspended', 'track_steps', 'tres_alloc', 'tres_req', 'job_type']

dcgm_job columns:
['id_job', 'dcgm_rows', 'energy_j', 'gpu_exec_s', 'sm_util_mean']

Scheduler dtypes:
id_job                  int64
id_array_job            int64
id_array_task           int64
id_user                 int64
kill_requid             int64
nodes_alloc             int64
nodelist                  str
cpus_req                int64
derived_ec              int64
exit_code               int64
gres_req                  str
gres_alloc                str
gres_used             float64
array_max_tasks         int64
array_task_pending      int6

### 2.Standardize time fields and create the executed-job population

In [4]:
df = sched_canon.copy()

# Replace negative placeholder values with missing values
time_cols = ["time_submit", "time_start", "time_end", "time_eligible", "time_suspended"]
for c in time_cols:
    if c in df.columns:
        df.loc[df[c] < 0, c] = np.nan

# Create core time-based features
df["queue_wait_s"] = df["time_start"] - df["time_submit"]
df["runtime_s"] = df["time_end"] - df["time_start"]
df["turnaround_s"] = df["time_end"] - df["time_submit"]

# Keep only executed jobs with valid and non-negative durations
exec_df = df[
    df["time_submit"].notna() &
    df["time_start"].notna() &
    df["time_end"].notna() &
    (df["queue_wait_s"] >= 0) &
    (df["runtime_s"] >= 0) &
    (df["turnaround_s"] >= 0)
].copy()

print("All canonical jobs:", len(df))
print("Executed jobs:", len(exec_df))
print("Executed share:", len(exec_df) / len(df))

All canonical jobs: 287069
Executed jobs: 217638
Executed share: 0.7581382873107162


### 3.Create a relative weekly time index

The dataset uses relative time values rather than wall-clock dates.
To compute weekly metrics consistently, we construct a relative week index from the earliest submit time in the executed-job population.

In [5]:
t0 = exec_df["time_submit"].min()
exec_df["t_rel_week"] = ((exec_df["time_submit"] - t0) // (86400 * 7)).astype(int)

print("Weeks covered:", exec_df["t_rel_week"].min(), "to", exec_df["t_rel_week"].max())

Weeks covered: 0 to 52


### 4.Build segmentation variables

In [6]:
# CPU request cleaning
# Keep original cpus_req, but use a cleaned version for bucket creation
exec_df["cpus_req_clean"] = exec_df["cpus_req"].replace(0, np.nan)

exec_df["cpu_bucket"] = pd.cut(
    exec_df["cpus_req_clean"],
    bins=[0, 1, 4, 16, 64, 1e9],
    labels=["1", "2-4", "5-16", "17-64", "65+"],
    include_lowest=True
)

exec_df["cpu_bucket"] = exec_df["cpu_bucket"].cat.add_categories(["unknown"]).fillna("unknown")

print(exec_df["cpu_bucket"].value_counts(normalize=True))
print(exec_df["partition"].value_counts(normalize=True))

cpu_bucket
1          0.577597
17-64      0.213754
5-16       0.096706
2-4        0.085546
65+        0.026397
unknown    0.000000
Name: proportion, dtype: float64
partition
normal    0.488127
gpu       0.466619
gaia      0.045231
test      0.000023
Name: proportion, dtype: float64


### 5.Others

In [7]:
# Job-level north star input
job_level_ns = exec_df[[
    "id_job",
    "t_rel_week",
    "partition",
    "job_type",
    "cpus_req",
    "cpus_req_clean",
    "cpu_bucket",
    "queue_wait_s",
    "runtime_s",
    "turnaround_s"
]].copy()

display(job_level_ns.head())

,id_job,t_rel_week,partition,job_type,cpus_req,cpus_req_clean,cpu_bucket,queue_wait_s,runtime_s,turnaround_s
0,61134,22,gpu,OTHER,1,1,1,0.0,1739.0,1739.0
1,246873,12,gpu,OTHER,1,1,1,15.0,0.0,15.0
2,325713,22,gaia,OTHER,2,2,2-4,0.0,27.0,27.0
3,1055011,40,gaia,OTHER,1,1,1,1.0,0.0,1.0
5,1363534,41,normal,OTHER,4,4,2-4,2.0,12904.0,12906.0


In [8]:
guardrail_fail = sched_canon.copy()

# Clean submit time for weekly grouping
guardrail_fail.loc[guardrail_fail["time_submit"] < 0, "time_submit"] = np.nan

# Relative week index for all jobs with valid submit time
t0_all = guardrail_fail["time_submit"].dropna().min()
guardrail_fail = guardrail_fail[guardrail_fail["time_submit"].notna()].copy()
guardrail_fail["t_rel_week"] = ((guardrail_fail["time_submit"] - t0_all) // (86400 * 7)).astype(int)

# Define completion status
guardrail_fail["is_completed"] = (guardrail_fail["state"] == 3)
guardrail_fail["is_not_completed"] = ~guardrail_fail["is_completed"]

display(guardrail_fail[["id_job", "state", "is_completed", "t_rel_week"]].head())

,id_job,state,is_completed,t_rel_week
0,61134,3,True,22
1,246873,5,False,12
2,325713,3,True,22
3,1055011,3,True,40
4,1218085,4,False,12


In [9]:
exec_jobs = exec_df.merge(dcgm_job, on="id_job", how="left")

# Define telemetry availability
if "sm_util_mean" in exec_jobs.columns:
    exec_jobs["gpu_observed"] = exec_jobs["sm_util_mean"].notna()
else:
    # fallback if your aggregated column is named differently
    possible_cols = [c for c in exec_jobs.columns if "sm" in c.lower() and "util" in c.lower()]
    print("Possible SM util columns:", possible_cols)
    exec_jobs["gpu_observed"] = exec_jobs[possible_cols[0]].notna()

print("GPU observed share:", exec_jobs["gpu_observed"].mean())
display(exec_jobs.head())

GPU observed share: 0.343864582471811


,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_req,gres_alloc,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,queue_wait_s,runtime_s,turnaround_s,t_rel_week,cpus_req_clean,cpu_bucket,dcgm_rows,energy_j,gpu_exec_s,sm_util_mean,gpu_observed
0,61134,41161693674,4595979483,41415979807,51671871839,1,['r7269000-n717709'],1,0,0,gpu:1,gpu:1,NaN,0,0,xeon-g6,8,51200,gpu,10362,3,240,13633668.0,13633668.0,13633668.0,13635407.0,NaN,0,"1=1,2=51200,4=1,5=1,1001=1","1=1,2=51200,4=1,5=1,1001=1",OTHER,0.0,1739.0,1739.0,22,1,1,NaN,NaN,NaN,NaN,False
1,246873,41161693674,4595979483,41415979807,51671871839,1,['r8532701-n772143'],1,256,256,gpu:1,gpu:1,NaN,0,0,xeon-g6,4,10240,gpu,10676,5,240,7825336.0,7825348.0,7825351.0,7825351.0,NaN,0,"1=1,2=10240,4=1,5=1,1001=1","1=1,2=10240,4=1,5=1,1001=1",OTHER,15.0,0.0,15.0,12,1,1,NaN,NaN,NaN,NaN,False
2,325713,41161693674,4595979483,21074900298,51671871839,1,['r9659953-n750018'],2,0,0,gpu:volta:2,gpu:2,NaN,0,0,xeon-g6,2,9223372036854785408,gaia,10317,3,4294967295,13416154.0,13416154.0,13416154.0,13416181.0,NaN,0,"1=2,2=19200,4=1,5=2,1002=2","1=2,2=19200,4=1,5=2,1002=2",OTHER,0.0,27.0,27.0,22,2,2-4,NaN,NaN,NaN,NaN,False
3,1055011,41161693674,4595979483,66088413977,51671871839,1,['r368967-n172107'],1,0,0,gpu:volta:1,gpu:1,NaN,0,0,xeon-g6,4,9223372036854785408,gaia,10213,3,5,24431626.0,24431626.0,24431627.0,24431627.0,NaN,0,"1=1,2=9600,4=1,5=1,1002=1","1=1,2=9600,4=1,5=1,1002=1",OTHER,1.0,0.0,1.0,40,1,1,1.0,0.0,0.55,0.0,True
4,1363534,42573057296,8110851898,65160314285,51671871839,1,['r368967-n750018'],4,0,0,gpu:volta:1,gpu:1,NaN,0,0,xeon-g6,4,9223372036854784308,normal,10276,3,21600,25087038.0,25087039.0,25087040.0,25099944.0,NaN,0,"1=4,2=34000,4=1,5=4,1002=1","1=4,2=34000,4=1,5=4,1002=1",OTHER,2.0,12904.0,12906.0,41,4,2-4,1.0,781482.0,12904.10,96.0,True


The next step is transforming raw telemetry units into human-readable units, as to enable better business interpretation and future analysis.

In [10]:
# Optional convenience fields
if "energy_j" in exec_jobs.columns:
    exec_jobs["energy_kwh"] = exec_jobs["energy_j"] / 3.6e6

if "gpu_exec_s" in exec_jobs.columns:
    exec_jobs["gpu_exec_h"] = exec_jobs["gpu_exec_s"] / 3600.0

if "sm_util_mean" in exec_jobs.columns and "gpu_exec_h" in exec_jobs.columns:
    exec_jobs["gpu_util_hours"] = (exec_jobs["sm_util_mean"] / 100.0) * exec_jobs["gpu_exec_h"]

## Selected Metrics

- **North Star Metric**
  - Weekly P95 Turnaround Time

- **Guardrail Metrics**
  - Failure / Non-completion Rate
  - Median GPU Utilization (GPU-observed jobs only)

This combination reflects real-world metric governance practices, where a primary success metric is protected by reliability and efficiency guardrails.


### 1.North Star Metric  
#### **Weekly P95 Turnaround Time**

**Definition**

For each executed job:

\[
\text{Turnaround Time} = \text{time\_end} - \text{time\_submit}
\]

The North Star metric is defined as the **95th percentile of turnaround time**, computed weekly over executed jobs.

---

**Why This Metric Was Chosen**

- It reflects **user-perceived time-to-result**
- It captures both queueing delay and execution time
- It focuses on tail behavior (P95), not averages
- It is sensitive to congestion and workload mix
- It has broad data coverage across jobs

---

**What This Metric Illustrates**

- Overall service quality
- Congestion and capacity stress
- Tail latency risk
- Temporal instability
- Sensitivity to workload segmentation

Because turnaround time is heavy-tailed, it is particularly well-suited for demonstrating why metric aggregation choices (mean vs percentile) matter for validity.



In [11]:
weekly_ns = job_level_ns.groupby("t_rel_week").agg(
    jobs=("id_job", "count"),
    p95_turnaround_s=("turnaround_s", lambda x: x.quantile(0.95)),
    p50_turnaround_s=("turnaround_s", "median"),
    p95_queue_wait_s=("queue_wait_s", lambda x: x.quantile(0.95)),
    p50_queue_wait_s=("queue_wait_s", "median"),
    p50_runtime_s=("runtime_s", "median")
).reset_index()

display(weekly_ns.head())

,t_rel_week,jobs,p95_turnaround_s,p50_turnaround_s,p95_queue_wait_s,p50_queue_wait_s,p50_runtime_s
0,0,962,71723.85,2620.0,58174.90,23.5,861.0
1,1,801,173279.00,1044.0,37448.00,2.0,633.0
2,2,589,97838.00,2556.0,28628.40,4.0,923.0
3,3,304,85750.45,2526.0,27128.75,1.0,754.0
4,4,539,67150.00,931.0,16444.00,4.0,526.0


### 2.Guardrail Metric 1  
#### Failure / Non-Completion Rate

**Definition**

The failure rate is defined as the share of jobs that do not complete successfully:

\[
\text{Failure Rate} = \frac{\text{jobs not completed}}{\text{total jobs}}
\]

Completion is inferred from scheduler state information in the canonical job table.

---

**Why This Metric Was Chosen**

- Prevents “gaming” of the North Star metric
- Ensures performance gains are not achieved by canceling or failing jobs
- Captures system reliability and robustness

---

**What This Metric Illustrates**

- Operational reliability
- State-driven missingness
- Segmentation bias across partitions or job types
- Drift in failure behavior over time




This metric protects against falsely interpreting improvements in service time as positive if they come at the cost of more job failures or non-completions.
It should be computed on the full canonical scheduler population, not only executed jobs.

**Assumption:** state==3 represents a completed job, whereas the other states-not completed.




In [12]:
weekly_fail = guardrail_fail.groupby("t_rel_week").agg(
    jobs=("id_job", "count"),
    failure_rate=("is_not_completed", "mean"),
    completion_rate=("is_completed", "mean")
).reset_index()

display(weekly_fail.head())

,t_rel_week,jobs,failure_rate,completion_rate
0,0,1107,0.345077,0.654923
1,1,852,0.393192,0.606808
2,2,648,0.455247,0.544753
3,3,323,0.470588,0.529412
4,4,625,0.505600,0.494400



### 3.Guardrail Metric 2  
#### Median GPU Utilization (GPU-Observed Jobs)

**Definition**

For jobs with available GPU telemetry, GPU utilization is measured using DCGM metrics. Job-level GPU utilization is aggregated and summarized weekly using the median.

---

**Why This Metric Was Chosen**

- GPUs represent a major cost driver
- Low utilization indicates inefficiency and waste
- Scheduler metadata alone overstates GPU usage
- Telemetry-based metrics allow stronger validity testing

---

**What This Metric Illustrates**

- Efficiency of hardware usage
- Gap between requested and actual GPU usage
- Segment-level inefficiencies (node, job size, workload)
- Impact of telemetry missingness

GPU utilization is only meaningful for jobs with DCGM telemetry.
We therefore merge job-level scheduler data with the aggregated GPU telemetry and define a GPU-observed population.

In [13]:
# Adjust the utilization column name if needed
util_col = "sm_util_mean"
if util_col not in exec_jobs.columns:
    util_candidates = [c for c in exec_jobs.columns if "sm" in c.lower() and "util" in c.lower()]
    util_col = util_candidates[0]

gpu_jobs = exec_jobs[exec_jobs["gpu_observed"]].copy()

weekly_gpu = gpu_jobs.groupby("t_rel_week").agg(
    gpu_jobs=("id_job", "count"),
    median_gpu_util=(util_col, "median"),
    p25_gpu_util=(util_col, lambda x: x.quantile(0.25)),
    p75_gpu_util=(util_col, lambda x: x.quantile(0.75))
).reset_index()

display(weekly_gpu.head())

,t_rel_week,gpu_jobs,median_gpu_util,p25_gpu_util,p75_gpu_util
0,35,2161,2.0,0.0,40.500
1,36,9013,1.0,0.0,16.000
2,37,3095,3.0,0.0,31.000
3,38,1950,1.0,0.0,32.000
4,39,1438,4.5,0.0,23.875


In [ ]:
# Merge all weekly metrics into a single DataFrame for easier comparison

weekly_metrics = (
    weekly_ns
    .merge(weekly_fail, on="t_rel_week", how="left", suffixes=("", "_fail"))
    .merge(weekly_gpu, on="t_rel_week", how="left")
)

display(weekly_metrics.head())

,t_rel_week,jobs,p95_turnaround_s,p50_turnaround_s,p95_queue_wait_s,p50_queue_wait_s,p50_runtime_s,jobs_fail,failure_rate,completion_rate,gpu_jobs,median_gpu_util,p25_gpu_util,p75_gpu_util
0,0,962,71723.85,2620.0,58174.90,23.5,861.0,1107,0.345077,0.654923,NaN,NaN,NaN,NaN
1,1,801,173279.00,1044.0,37448.00,2.0,633.0,852,0.393192,0.606808,NaN,NaN,NaN,NaN
2,2,589,97838.00,2556.0,28628.40,4.0,923.0,648,0.455247,0.544753,NaN,NaN,NaN,NaN
3,3,304,85750.45,2526.0,27128.75,1.0,754.0,323,0.470588,0.529412,NaN,NaN,NaN,NaN
4,4,539,67150.00,931.0,16444.00,4.0,526.0,625,0.505600,0.494400,NaN,NaN,NaN,NaN



### Why These Metrics Work Well Together

| Metric | Role | Illustrates |
|------|------|-----------|
| P95 Turnaround Time | North Star | User experience and tail latency |
| Failure Rate | Guardrail | Reliability and integrity |
| GPU Utilization | Guardrail | Efficiency and cost discipline |

Together, these metrics enable:

- Trade-off analysis between speed, reliability, and efficiency
- Detection of segmentation bias
- Stability and drift analysis
- Reliability and variance estimation
- CUPED-based variance reduction demonstrations

---

#### Final Framing

This project does not aim to optimize the datacenter.

Instead, it demonstrates how to **rigorously evaluate the validity of operational metrics** in a real-world infrastructure system. The focus is on understanding whether metrics are:

- Statistically reliable
- Stable over time
- Fair across segments
- Resistant to missing data
- Sensitive to meaningful changes
- Appropriate for decision-making

The dataset provides a realistic environment for studying metric governance in practice.

| Concept           | North Star        | Failure Rate | GPU Util |
| ----------------- | ----------------- | ------------ | -------- |
| Missingness       | ✔                 | ✔            | ✔        |
| Drift             | ✔                 | ✔            | ✔        |
| Stability         | ✔                 |              | ✔        |
| Reliability       | ✔ (bootstrap P95) |              |          |
| Segmentation bias | ✔                 | ✔            | ✔        |
| Sensitivity       | ✔                 |              |          |
| CUPED             | ✔                 | ✔            | ✔        |



**Goal:** to demonstrate how to rigorously evaluate the validity of operational metrics in a real-world infrastructure system.

**Why Not Use Mean?**

All three candidate metrics are heavy-tailed.

Mean would:

- Be unstable

- Be dominated by rare outliers

- Be misleading for operational decisions

This makes them ideal examples for metric validity education.

## Save datasets

In [15]:
# Job-level datasets
job_level_ns.to_parquet(PROCESSED_DIR / "job_level_northstar_inputs.parquet", index=False)
guardrail_fail.to_parquet(PROCESSED_DIR / "job_level_failure_inputs.parquet", index=False)
exec_jobs.to_parquet(PROCESSED_DIR / "job_level_metrics_joined.parquet", index=False)

# Weekly datasets
weekly_ns.to_csv(PROCESSED_DIR / "weekly_northstar_metrics.csv", index=False)
weekly_fail.to_csv(PROCESSED_DIR / "weekly_failure_metrics.csv", index=False)
weekly_gpu.to_csv(PROCESSED_DIR / "weekly_gpu_metrics.csv", index=False)
weekly_metrics.to_csv(PROCESSED_DIR / "weekly_metrics_combined.csv", index=False)

print("Saved files to:", PROCESSED_DIR.resolve())
print("\nProcessed files:")
for p in sorted(PROCESSED_DIR.glob("*")):
    print("-", p.name)

Saved files to: /Users/newuser/Desktop/Victoria/Projects_2025/metric-validation-playbook/data/processed

Processed files:
- job_level_failure_inputs.parquet
- job_level_metrics_joined.parquet
- job_level_northstar_inputs.parquet
- weekly_failure_metrics.csv
- weekly_gpu_metrics.csv
- weekly_metrics_combined.csv
- weekly_northstar_metrics.csv
